# Colab runner — poisoning efficiency, LoRA vs. full fine-tuning

Runs the **same code and configs** as the GAIVI Slurm scripts, on a Colab GPU.
Use it only as declared in the README/report: *experiments ran on a Colab T4 because the GAIVI
`ClsGenAI` queue was full*. Every run records the GPU in `env.json`.

**Rules**
1. Runtime → Change runtime type → **T4 GPU** (or better). Keep ONE GPU type for the whole main comparison.
2. Results are written straight to Google Drive (`MyDrive/backdoor_results/`), so a disconnect loses at most
   the run in progress. Re-running a sweep cell skips finished runs (`--skip-existing`).
3. Cells 1–5 are cheap. **Do not run the sweep cell (6) before the cost has been reviewed.**

In [ ]:
# 1. GPU check — expect "Tesla T4" (fp16 only, no bf16; same code path as the TITAN RTX)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Google Drive (results live here) + clone/update the repo.
# The repo is private: add a GitHub fine-grained token (read-only, this repo only) in
# Colab's left sidebar -> key icon "Secrets" -> name GH_TOKEN -> enable notebook access.
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/backdoor_results'
os.makedirs(f'{RESULTS}/runs', exist_ok=True); os.makedirs(f'{RESULTS}/logs', exist_ok=True)

REPO = 'github.com/Hansika-s24/backdoor-poisoning-efficiency.git'
if not os.path.exists('/content/backdoor'):
    tok = userdata.get('GH_TOKEN')
    !git clone -q https://{tok}@{REPO} /content/backdoor
    !git -C /content/backdoor remote set-url origin https://{REPO}   # don't leave the token in .git/config
    del tok
else:
    !git -C /content/backdoor pull -q
%cd /content/backdoor
!git log --oneline -1

In [ ]:
# 3. Pinned environment (same versions as GAIVI). Scripts run as `!python ...` subprocesses,
# so Colab's preinstalled numpy/torch in this kernel don't matter. ~3-5 min.
!pip -q install torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip -q install -r requirements.txt
!python -c "import torch, transformers, peft; print(torch.__version__, torch.cuda.get_device_name(0), transformers.__version__, peft.__version__)"

In [ ]:
# 4. Environment check + unit tests + smoke test (~5 min). Same as scripts/slurm/check_env.sbatch.
!python -m pytest -q 2>&1 | tail -3
!python scripts/check_env.py --model Qwen/Qwen2.5-0.5B 2>&1 | tee {RESULTS}/logs/check_env.log | tail -25
!python scripts/smoke_test.py 2>&1 | tee {RESULTS}/logs/smoke.log | tail -5

In [ ]:
# 5. Timing probe: LoRA r=8, seed 0, n=0 and n=50 (2 runs). Measures real sec/step and
# checks whether the attack works at all at 1% poison. Paste the printed summary to Claude.
RUNS = f'{RESULTS}/runs'
for n in [0, 50]:
    !python scripts/run_experiment.py --config configs/base.yaml --skip-existing         --set results_root={RUNS} --set seed=0 --set attack.n_poison={n} 2>&1 | tee -a {RESULTS}/logs/timing_probe.log | tail -4

import json, glob
for f in sorted(glob.glob(f'{RUNS}/*/metrics.json')):
    m = json.load(open(f))
    keep = ['n_poison','seed','asr','asr_exact','false_trigger_rate','clean_loss','rougeL',
            'spectral_auroc','train_time_s','sec_per_step','peak_gpu_mem_gb_train']
    print(f.split('/')[-2], {k: m.get(k) for k in keep})

### 6. Sweeps — run only after the cost has been reviewed

One cell run = one block. Set `EXTRA` for the arm:

| Block | `EXTRA` |
|---|---|
| Main curve, LoRA r=8 | `[]` |
| Rank ablation | `['arm.rank=16']` (etc.) |
| Full fine-tuning (LR chosen on val first) | `['arm.type=full', 'train.lr=<chosen>', 'save_adapter=false']` — if it runs out of memory on 16 GB, add `'train.micro_batch_size=4', 'train.grad_accum=4'` (same effective batch 16) and use that for **every** full-FT run |

In [ ]:
SEEDS = [0, 1, 2]
NS = [0, 1, 2, 5, 10, 20, 50, 100, 200]
EXTRA = []          # see table above
sets = ' '.join(f'--set {kv}' for kv in EXTRA)
for seed in SEEDS:
    for n in NS:
        print(f'=== seed={seed} n={n} extra={EXTRA} ===')
        !python scripts/run_experiment.py --config configs/base.yaml --skip-existing             --set results_root={RUNS} --set seed={seed} --set attack.n_poison={n} {sets} 2>&1 | tee -a {RESULTS}/logs/sweep.log | tail -3

In [ ]:
# 7. Summaries (CSV files go to Drive next to the runs)
!python scripts/summarize.py --runs {RUNS} --out {RESULTS}
!cat {RESULTS}/n90.csv